<a href="https://colab.research.google.com/github/ragiokay/AI_transcribe/blob/main/AI_transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 錄音逐字稿處理 Pipeline v2

## 流程說明
```
Step 1  │ Whisper 轉錄              → {BASE_NAME}_1_原始逐字稿.txt
Step 2  │ Gemini 音節修復 (保留冗詞) → {BASE_NAME}_2_音節修復保留冗詞版.md
Step 3  │ ⚠️ 人工聽音修復 (手動)
Step 4  │ Gemini 刪除冗詞           → {BASE_NAME}_4_刪除冗詞版.md
Step 5  │ ⚠️ 人工確認 (手動)
Step 6  │ Gemini 時間序局部摘要      → {BASE_NAME}_6_時間序局部摘要.md
Step 7  │ ⚠️ 人工補充摘要 (手動)
Step 8  │ Gemini 最終重構摘要        → {BASE_NAME}_8_最終重構摘要.md
```

## 使用方式
1. 修改下方 Cell 0 的 `BASE_NAME` 與 `MEETING_CONTEXT`
2. 確保音檔已放到 Google Drive 的 `AI_transcribe` 資料夾
3. 依序執行各 Cell，遇到人工步驟時請先處理再繼續

In [13]:
# ============================================================
# Cell 0：【每次使用前修改這裡】專案設定
# ============================================================

# 檔案基本名稱（音檔需命名為 {BASE_NAME}.m4a 或 .mp3）
BASE_NAME = "NLP"

# 這場會議的背景脈絡（讓 Gemini 修復時更準確）
MEETING_CONTEXT = """
這是一場資工所「NLP 期末專案」的進度報告與助教討論。
專案主題：情緒語氣是否會干擾 LLM 的事實查核（Fact-Checking）邏輯判斷。
專有名詞清單：Ground Truth、Baseline、Few-shot Prompting、LLM、Prompt、
CoFED（Cofacts）資料集、Fake、True、Neutral、Flip（翻轉）、
Atomic Facts、Semantic Similarity、Binomial Test（二項式檢定）、
p-value、Rule-based、Hallucination（幻覺）、NVIDIA、
Public Health、Scam、Category、Confidence Score。
語者：主要有學生（報告者）與助教（提問者）。
"""

# Gemini 模型選擇（優先使用 gemini-2.5-flash，穩定且便宜）
# 可換成 "gemini-2.5-pro" 若需要更高品質但費用較高
LLM_MODEL_NAME = "gemini-2.5-flash"

# Google Drive 路徑設定
DRIVE_DIR = f"/content/drive/MyDrive/AI_transcribe"

print(f"✅ 設定完成")
print(f"   BASE_NAME    : {BASE_NAME}")
print(f"   LLM Model    : {LLM_MODEL_NAME}")
print(f"   Drive Dir    : {DRIVE_DIR}")

✅ 設定完成
   BASE_NAME    : NLP
   LLM Model    : gemini-2.5-flash
   Drive Dir    : /content/drive/MyDrive/AI_transcribe


In [14]:
# ============================================================
# Cell 1：環境安裝與初始化（每次 runtime 重啟後需執行）
# ============================================================

!pip install -q faster-whisper google-generativeai

import os
import time
import textwrap
from google.colab import drive, userdata
import google.generativeai as genai

# 掛載 Drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

# 定義所有檔案路徑
AUDIO_EXTENSIONS = [".m4a", ".mp3", ".wav", ".ogg"]
AUDIO_FILE_PATH = None
for ext in AUDIO_EXTENSIONS:
    candidate = os.path.join(DRIVE_DIR, f"{BASE_NAME}{ext}")
    if os.path.exists(candidate):
        AUDIO_FILE_PATH = candidate
        break

PATH_1_RAW      = os.path.join(DRIVE_DIR, f"{BASE_NAME}_1_原始逐字稿.txt")
PATH_2_REPAIR   = os.path.join(DRIVE_DIR, f"{BASE_NAME}_2_音節修復保留冗詞版.md")
PATH_4_CLEAN    = os.path.join(DRIVE_DIR, f"{BASE_NAME}_4_刪除冗詞版.md")
PATH_6_CHRONO   = os.path.join(DRIVE_DIR, f"{BASE_NAME}_6_時間序局部摘要.md")
PATH_8_FINAL    = os.path.join(DRIVE_DIR, f"{BASE_NAME}_8_最終重構摘要.md")

# 設定 Gemini API
# 請在 Colab 左側 🔑 Secrets 新增 GEMINI_API_KEY
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

print(f"✅ 環境初始化完成")
if AUDIO_FILE_PATH:
    print(f"   🎵 找到音檔: {AUDIO_FILE_PATH}")
else:
    print(f"   ⚠️  找不到音檔，請確認 {DRIVE_DIR} 內有 {BASE_NAME}.m4a（或 .mp3）")

# 輔助函式：呼叫 Gemini（含 retry 與錯誤處理）
def call_gemini(system_prompt, user_text, temperature=0.1, max_retries=3):
    """呼叫 Gemini API，自動重試，回傳回應文字。"""
    model = genai.GenerativeModel(
        model_name=LLM_MODEL_NAME,
        system_instruction=system_prompt
    )
    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                user_text,
                generation_config=genai.GenerationConfig(
                    temperature=temperature,
                    max_output_tokens=65536
                )
            )
            return response.text
        except Exception as e:
            print(f"   ⚠️  嘗試 {attempt+1}/{max_retries} 失敗: {e}")
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1))
    raise RuntimeError("Gemini API 多次失敗，請檢查 API Key 與模型名稱。")

# 輔助函式：長文字分塊處理（避免超過 context window）
def chunk_text(text, max_chars=60000):
    """將長文字依時間標記分塊，避免切斷同一段落。"""
    if len(text) <= max_chars:
        return [text]
    lines = text.split('\n')
    chunks, current_chunk, current_len = [], [], 0
    for line in lines:
        if current_len + len(line) > max_chars and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk, current_len = [], 0
        current_chunk.append(line)
        current_len += len(line) + 1
    if current_chunk:
        chunks.append('\n'.join(current_chunk))
    print(f"   📦 文字太長，已分成 {len(chunks)} 塊處理")
    return chunks

def process_with_chunking(system_prompt, text, temperature=0.1):
    """分塊呼叫 Gemini 並合併結果。"""
    chunks = chunk_text(text)
    results = []
    for i, chunk in enumerate(chunks):
        print(f"   🔄 處理第 {i+1}/{len(chunks)} 塊...")
        prompt = f"（這是第 {i+1}/{len(chunks)} 段，請依指示處理）\n\n{chunk}"
        results.append(call_gemini(system_prompt, prompt, temperature))
        if len(chunks) > 1:
            time.sleep(2)  # 避免 rate limit
    return '\n\n---（分塊接續）---\n\n'.join(results) if len(chunks) > 1 else results[0]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 環境初始化完成
   🎵 找到音檔: /content/drive/MyDrive/AI_transcribe/NLP.m4a


In [16]:
# ============================================================
# Cell 2：[Step 1] Whisper 強效轉錄
# ============================================================

if os.path.exists(PATH_1_RAW):
    char_count = len(open(PATH_1_RAW, encoding='utf-8').read())
    print(f"✅ 已有 [1_原始逐字稿]，跳過轉錄（{char_count:,} 字）")
    print(f"   路徑: {PATH_1_RAW}")
else:
    if not AUDIO_FILE_PATH:
        raise FileNotFoundError(f"找不到音檔！請確認 {DRIVE_DIR} 內有 {BASE_NAME}.m4a")

    print("🚀 開始 Whisper large-v3 轉錄...")
    from faster_whisper import WhisperModel

    model = WhisperModel("large-v3", device="cuda", compute_type="float16")

    # 保持跟原始可運作版本一樣的最簡設定，不加任何額外參數
    segments, info = model.transcribe(
        AUDIO_FILE_PATH,
        beam_size=5,
        language="zh"
    )
    print(f"📊 偵測語言: {info.language}（信心度: {info.language_probability:.2f}）")

    raw_text = ""
    seg_count = 0
    for segment in segments:
        line = f"[{segment.start:.1f}s - {segment.end:.1f}s] {segment.text.strip()}\n"
        raw_text += line
        seg_count += 1
        print(line.strip())  # 全部即時印出，方便確認有沒有跳段

    with open(PATH_1_RAW, "w", encoding="utf-8") as f:
        f.write(raw_text)

    print(f"\n💾 [Step 1 完成] 原始逐字稿已儲存")
    print(f"   共 {seg_count} 段，{len(raw_text):,} 字")
    print(f"   路徑: {PATH_1_RAW}")

🚀 開始 Whisper large-v3 轉錄...
📊 偵測語言: zh（信心度: 1.00）
[0.0s - 5.0s] 我們會先講一下我們的實驗流程
[5.0s - 8.0s] 我們是用Cofed這個資料集
[8.0s - 11.0s] 然後我們會給大家做一些資料的過濾
[11.0s - 14.0s] 就是它如果這個資料集太長太短會掉
[14.0s - 16.0s] 然後會取得原始資料
[16.0s - 19.0s] 然後我們會用LLM來把它做改寫
[19.0s - 22.0s] 我們是要做改寫語氣之後的判斷
[22.0s - 24.0s] 就是這個改寫語氣之後
[24.0s - 26.0s] 會不會影響到LLM的判斷
[26.0s - 29.0s] 所以我們會先把這邊得到原始訊息
[29.0s - 32.0s] 然後再給大家做一個完結
[32.0s - 34.0s] 然後之後用情緒分析
[34.0s - 36.0s] 好這是資料集
[36.0s - 38.0s] 這次要解內容
[38.0s - 40.0s] 這會不會有原始訊息
[40.0s - 43.0s] 會有原始訊息
[43.0s - 45.0s] 事實查核
[45.0s - 48.0s] 那你們跟情緒的關係是怎樣
[48.0s - 51.0s] 後面會提到
[51.0s - 53.0s] 所以我們要改寫
[53.0s - 54.0s] 就是我們要加入情緒
[54.0s - 58.0s] 我想到你們是說要把原本有一個資料集
[58.0s - 59.0s] 要把它加情緒
[59.0s - 61.0s] 然後看它有沒有判斷
[61.0s - 62.0s] 或會不會跟原始
[62.0s - 64.0s] 就原本只判斷原始的
[64.0s - 65.0s] 跟加入情緒的時候
[65.0s - 67.0s] 看它會不會不一樣
[67.0s - 69.0s] 然後這是它裡面的Fill
[69.0s - 71.0s] 然後它會有它的標籤
[71.0s - 72.0s] 然後因為原始訊息
[72.0s - 76.0s] 然後這邊有查核之後的訊息
[76.0s - 79.0s] 然後我們是用LLM來改寫
[79.0s - 82.0s] 我們改寫的話我們就選擇六種語氣
[82.0s - 85.0s] 然後我們的目標是要保